# AI-Powered Movie Recommendation Demo

This notebook demonstrates the completed personalized movie recommendation
system.

The system combines:

- content-based recommendation
- Pearson collaborative filtering
- k-means user segmentation
- hybrid recommendation scoring
- personalized recommendation explanations

Select a MovieLens user and the system will analyze their historical
preferences and generate personalized movie recommendations.

In [1]:
import numpy as np
import pandas as pd
import ipywidgets as widgets

from IPython.display import (
    display,
    clear_output,
    HTML
)

# Reuse the tested engine classes from notebook 02's support module.
from recommender import (
    ContentBasedRecommender,
    CollaborativeRecommender,
    ClusterRecommender,
    HybridRecommender,
    RecommendationExplainer
)

In [2]:
# Load the two processed artifacts created by the preceding notebooks.
movie_ratings = pd.read_csv(
    "data/processed/movie_ratings.csv"
)

user_clusters = pd.read_csv(
    "data/processed/user_clusters.csv"
)

print(
    f"Ratings: {len(movie_ratings):,}"
)

print(
    f"Users: {movie_ratings['userId'].nunique():,}"
)

print(
    f"Movies: {movie_ratings['movieId'].nunique():,}"
)

Ratings: 100,836
Users: 610
Movies: 9,724


In [3]:
# Global averages and rating counts provide quality/popularity signals.
movie_stats = (
    movie_ratings
    .groupby(
        [
            "movieId",
            "title"
        ],
        as_index=False
    )
    .agg(
        average_rating=(
            "rating",
            "mean"
        ),
        rating_count=(
            "rating",
            "count"
        )
    )
)

movie_stats.head()

,movieId,title,average_rating,rating_count
0,1,Toy Story (1995),3.920930,215
1,2,Jumanji (1995),3.431818,110
2,3,Grumpier Old Men (1995),3.259615,52
3,4,Waiting to Exhale (1995),2.357143,7
4,5,Father of the Bride Part II (1995),3.071429,49


In [4]:
# Collapse rating rows into the one-row-per-movie catalog expected by
# every recommendation engine.
movie_catalog = (
    movie_ratings[
        [
            "movieId",
            "title",
            "genres",
            "director",
            "release_year",
            "runtime",
            "tmdb_rating",
            "popularity"
        ]
    ]
    .drop_duplicates(
        subset="movieId"
    )
    .merge(
        movie_stats,
        on=[
            "movieId",
            "title"
        ],
        how="left"
    )
)

movie_catalog.head()

,movieId,title,genres,director,release_year,runtime,tmdb_rating,popularity,average_rating,rating_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,John Lasseter,1995.0,81.0,8.000,41.0572,3.920930,215
1,3,Grumpier Old Men (1995),Comedy|Romance,Howard Deutch,1995.0,101.0,6.466,5.9315,3.259615,52
2,6,Heat (1995),Action|Crime|Thriller,Michael Mann,1995.0,170.0,7.941,23.1580,3.946078,102
3,47,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,David Fincher,1995.0,127.0,8.380,33.8713,3.975369,203
4,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,Bryan Singer,1995.0,106.0,8.165,21.1306,4.237745,204


In [5]:
# All engines share the same immutable input snapshots; the hybrid engine
# coordinates their ranked candidate lists.
content_engine = ContentBasedRecommender(
    movie_ratings,
    movie_catalog
)

collaborative_engine = CollaborativeRecommender(
    movie_ratings,
    movie_catalog
)

cluster_engine = ClusterRecommender(
    movie_ratings,
    movie_catalog,
    user_clusters
)

hybrid_engine = HybridRecommender(
    movie_ratings,
    movie_catalog,
    content_engine,
    collaborative_engine,
    cluster_engine
)

explainer = RecommendationExplainer(
    movie_ratings
)

print(
    "Recommendation system loaded successfully."
)

Recommendation system loaded successfully.


## Personalized Recommendation Demo

The function below provides a simple user-facing interface for the completed
recommendation system.

For a selected MovieLens user, it displays:

- number of movies rated
- average rating
- k-means preference cluster
- strongest genre preferences
- several highly rated movies from the user's history
- final hybrid recommendations
- personalized explanations for each recommendation

In [6]:
def recommend_movies_for_user(
    user_id,
    n=5
):
    """
    Display personalized movie recommendations
    for an existing MovieLens user.
    """

    # --------------------------------------
    # 1. Validate user
    # --------------------------------------

    try:
        hybrid_engine.validate_user(
            user_id
        )

    except ValueError as error:
        print(error)
        return None


    # --------------------------------------
    # 2. User rating history
    # --------------------------------------

    user_history = (
        movie_ratings[
            movie_ratings["userId"] == user_id
        ]
        .copy()
    )


    ratings_count = len(
        user_history
    )

    average_rating = (
        user_history["rating"]
        .mean()
    )


    # --------------------------------------
    # 3. User cluster
    # --------------------------------------

    user_cluster = (
        cluster_engine
        .get_user_cluster(
            user_id
        )
    )


    # --------------------------------------
    # 4. Genre preferences
    # --------------------------------------

    genre_preferences = (
        content_engine
        .get_genre_preferences(
            user_id
        )
        .copy()
    )


    top_genres = (
        genre_preferences
        .sort_values(
            [
                "average_rating",
                "movies_rated"
            ],
            ascending=[
                False,
                False
            ]
        )
        .head(5)
    )


    # --------------------------------------
    # 5. Favorite movies
    # --------------------------------------

    favorite_movies = (
        user_history
        .sort_values(
            [
                "rating",
                "title"
            ],
            ascending=[
                False,
                True
            ]
        )
        .head(5)
    )


    # --------------------------------------
    # 6. Generate recommendations
    # --------------------------------------

    recommendations = (
        hybrid_engine
        .recommend(
            user_id=user_id,
            n=n
        )
        .copy()
    )


    # --------------------------------------
    # 7. Generate explanations
    # --------------------------------------

    recommendations[
        "explanation"
    ] = (
        recommendations
        .apply(
            lambda row:
            explainer.generate_explanation(
                user_id=user_id,
                recommendation=row
            ),
            axis=1
        )
    )


    # --------------------------------------
    # 8. Display user profile
    # --------------------------------------

    print("=" * 75)

    print(
        f"PERSONALIZED MOVIE RECOMMENDATIONS "
        f"FOR USER {user_id}"
    )

    print("=" * 75)


    print(
        f"\nMovies rated: "
        f"{ratings_count}"
    )

    print(
        f"Average rating: "
        f"{average_rating:.2f}/5"
    )

    print(
        f"Preference cluster: "
        f"{user_cluster}"
    )


    # --------------------------------------
    # 9. Display genre preferences
    # --------------------------------------

    print(
        "\nTOP GENRE PREFERENCES"
    )

    print("-" * 75)


    for genre, row in (
        top_genres.iterrows()
    ):

        print(
            f"{genre:<15} "
            f"{row['average_rating']:.2f}/5 "
            f"({int(row['movies_rated'])} movies)"
        )


    # --------------------------------------
    # 10. Display favorite movies
    # --------------------------------------

    print(
        "\nSOME HIGHLY RATED MOVIES"
    )

    print("-" * 75)


    for _, movie in (
        favorite_movies.iterrows()
    ):

        print(
            f"{movie['title']} "
            f"— {movie['rating']:.1f}/5"
        )


    # --------------------------------------
    # 11. Display recommendations
    # --------------------------------------

    print(
        "\nTOP RECOMMENDATIONS"
    )

    print("=" * 75)


    for position, (_, movie) in enumerate(
        recommendations.iterrows(),
        start=1
    ):

        print(
            f"\n{position}. "
            f"{movie['title']}"
        )

        print(
            f"   Genres: "
            f"{movie['genres']}"
        )

        print(
            f"   Director: "
            f"{movie['director']}"
        )

        print(
            f"   Hybrid score: "
            f"{movie['hybrid_score']:.3f}"
        )

        print(
            f"   Models supporting: "
            f"{int(movie['models_recommending'])}/3"
        )

        print(
            f"   Why: "
            f"{movie['explanation']}"
        )


    print(
        "\n" + "=" * 75
    )


    return recommendations

In [7]:
USER_ID = 1
NUMBER_OF_RECOMMENDATIONS = 5

In [8]:
final_recommendations = (
    recommend_movies_for_user(
        user_id=USER_ID,
        n=NUMBER_OF_RECOMMENDATIONS
    )
)

PERSONALIZED MOVIE RECOMMENDATIONS FOR USER 1

Movies rated: 232
Average rating: 4.37/5
Preference cluster: 2

TOP GENRE PREFERENCES
---------------------------------------------------------------------------
Film-Noir       5.00/5 (1 movies)
Animation       4.69/5 (29 movies)
Musical         4.68/5 (22 movies)
Children        4.55/5 (42 movies)
Drama           4.53/5 (68 movies)

SOME HIGHLY RATED MOVIES
---------------------------------------------------------------------------
Adventures of Robin Hood, The (1938) — 5.0/5
Alice in Wonderland (1951) — 5.0/5
All Quiet on the Western Front (1930) — 5.0/5
American Beauty (1999) — 5.0/5
American History X (1998) — 5.0/5

TOP RECOMMENDATIONS

1. Shawshank Redemption, The (1994)
   Genres: Crime|Drama
   Director: Frank Darabont
   Hybrid score: 0.998
   Models supporting: 3/3
   Why: Since you rated Goodfellas (1990) 5.0/5 and it shares Crime and Drama elements, you might enjoy Shawshank Redemption, The (1994). It is supported by all three

In [9]:
USER_ID = 50

recommend_movies_for_user(
    user_id=USER_ID,
    n=5
)

PERSONALIZED MOVIE RECOMMENDATIONS FOR USER 50

Movies rated: 310
Average rating: 2.78/5
Preference cluster: 0

TOP GENRE PREFERENCES
---------------------------------------------------------------------------
Film-Noir       3.50/5 (2 movies)
War             3.45/5 (11 movies)
Mystery         3.23/5 (20 movies)
Western         3.17/5 (6 movies)
Documentary     3.14/5 (14 movies)

SOME HIGHLY RATED MOVIES
---------------------------------------------------------------------------
2001: A Space Odyssey (1968) — 4.5/5
8 1/2 (8½) (1963) — 4.5/5
Apocalypse Now (1979) — 4.5/5
Lawrence of Arabia (1962) — 4.5/5
Akira (1988) — 4.0/5

TOP RECOMMENDATIONS

1. Rear Window (1954)
   Genres: Mystery|Thriller
   Director: Alfred Hitchcock
   Hybrid score: 0.928
   Models supporting: 3/3
   Why: Since you rated Vertigo (1958) 4.0/5 and it shares Mystery and Thriller elements, you might enjoy Rear Window (1954). It is supported by all three recommendation models: your content preferences, similar user

,movieId,title,genres,director,hybrid_score,models_recommending,content_score,collaborative_score,cluster_score,average_rating,rating_count,explanation
33,904,Rear Window (1954),Mystery|Thriller,Alfred Hitchcock,0.928,3,3.701579,4.744653,4.339286,4.261905,84,Since you rated Vertigo (1958) 4.0/5 and it sh...
81,1250,"Bridge on the River Kwai, The (1957)",Adventure|Drama|War,David Lean,0.901,3,3.763675,4.801017,4.181818,4.122222,45,Since you rated Lawrence of Arabia (1962) 4.5/...
91,1276,Cool Hand Luke (1967),Drama,Stuart Rosenberg,0.827,3,3.614731,4.619349,4.363636,4.271930,57,"Since you rated Mirror, The (Zerkalo) (1975) 4..."
67,1221,"Godfather: Part II, The (1974)",Crime|Drama,Francis Ford Coppola,0.821,3,3.727161,4.495157,4.423077,4.259690,129,"Since you rated Godfather, The (1972) 4.0/5 an..."
36,912,Casablanca (1942),Drama|Romance,Michael Curtiz,0.814,3,3.540764,4.664865,4.666667,4.240000,100,Since you rated Jules and Jim (Jules et Jim) (...


In [10]:
USER_ID = 200

recommend_movies_for_user(
    user_id=USER_ID,
    n=5
)

PERSONALIZED MOVIE RECOMMENDATIONS FOR USER 200

Movies rated: 334
Average rating: 3.81/5
Preference cluster: 2

TOP GENRE PREFERENCES
---------------------------------------------------------------------------
Film-Noir       4.67/5 (3 movies)
Mystery         4.35/5 (13 movies)
Musical         4.12/5 (16 movies)
Crime           4.05/5 (47 movies)
IMAX            4.00/5 (12 movies)

SOME HIGHLY RATED MOVIES
---------------------------------------------------------------------------
10 Things I Hate About You (1999) — 5.0/5
300 (2007) — 5.0/5
50 First Dates (2004) — 5.0/5
Across the Universe (2007) — 5.0/5
Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001) — 5.0/5

TOP RECOMMENDATIONS

1. Godfather, The (1972)
   Genres: Crime|Drama
   Director: Francis Ford Coppola
   Hybrid score: 0.855
   Models supporting: 3/3
   Why: Since you rated Shawshank Redemption, The (1994) 5.0/5 and it shares Crime and Drama elements, you might enjoy Godfather, The (1972). It is supported by all three re

,movieId,title,genres,director,hybrid_score,models_recommending,content_score,collaborative_score,cluster_score,average_rating,rating_count,explanation
55,858,"Godfather, The (1972)",Crime|Drama,Francis Ford Coppola,0.855333,3,4.095519,4.793606,4.379630,4.289062,192,"Since you rated Shawshank Redemption, The (199..."
99,1221,"Godfather: Part II, The (1974)",Crime|Drama,Francis Ford Coppola,0.800000,3,4.081812,4.690713,4.325581,4.259690,129,"Since you rated Shawshank Redemption, The (199..."
9,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,Bryan Singer,0.770000,3,4.116849,4.567810,4.270492,4.237745,204,Since you rated Kiss Kiss Bang Bang (2005) 5.0...
192,79132,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,Christopher Nolan,0.769000,3,4.078716,4.732414,4.272727,4.066434,143,Since you rated Fight Club (1999) 5.0/5 and it...
176,48516,"Departed, The (2006)",Crime|Drama|Thriller,Martin Scorsese,0.739667,3,4.049021,4.621662,4.333333,4.252336,107,Since you rated Layer Cake (2004) 4.5/5 and it...


## Recommendations for a New User

The system can also generate recommendations for someone who is not already
part of the MovieLens dataset.

The new user provides ratings for several movies they have seen. These ratings
are then used to:

- infer genre preferences
- identify similar MovieLens users
- assign the new user to the closest k-means preference cluster
- generate personalized hybrid recommendations
- produce human-readable recommendation explanations

This simulates the onboarding process of a real recommendation platform.

In [11]:
def search_movies(
    search_text,
    n=10
):
    """
    Search the movie catalog by title.
    """

    matches = (
        movie_catalog[
            movie_catalog["title"]
            .str.contains(
                search_text,
                case=False,
                na=False
            )
        ][
            [
                "movieId",
                "title",
                "genres"
            ]
        ]
        .head(n)
    )

    return matches

In [12]:
search_movies(
    "matrix"
)

,movieId,title,genres
166,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller
768,6365,"Matrix Reloaded, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX
772,6934,"Matrix Revolutions, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX
3071,27660,"Animatrix, The (2003)",Action|Animation|Drama|Sci-Fi


In [13]:
search_movies(
    "godfather"
)

,movieId,title,genres
836,2023,"Godfather: Part III, The (1990)",Crime|Drama|Mystery|Thriller
1027,858,"Godfather, The (1972)",Crime|Drama
1136,1221,"Godfather: Part II, The (1974)",Crime|Drama
3124,172591,The Godfather Trilogy: 1972-1990 (1992),(no genres listed)
5526,8607,Tokyo Godfathers (2003),Adventure|Animation|Drama


In [14]:
# Example onboarding profile. Keys are MovieLens IDs and values are
# explicit ratings on the dataset's 0.5-to-5.0 scale.
NEW_USER_RATINGS = {
    2571: 5.0,   # The Matrix
    2959: 5.0,   # Fight Club
    79132: 4.5,  # Inception
    260: 4.5,    # Star Wars
    1196: 4.5,   # Empire Strikes Back
    296: 4.0,    # Pulp Fiction
    318: 5.0,    # Shawshank Redemption
    356: 3.5,    # Forrest Gump
    480: 3.0,    # Jurassic Park
    1721: 2.0    # Titanic
}

In [15]:
selected_movies = (
    movie_catalog[
        movie_catalog["movieId"]
        .isin(
            NEW_USER_RATINGS.keys()
        )
    ][
        [
            "movieId",
            "title",
            "genres"
        ]
    ]
    .copy()
)

selected_movies[
    "new_user_rating"
] = (
    selected_movies["movieId"]
    .map(
        NEW_USER_RATINGS
    )
)

selected_movies.sort_values(
    "new_user_rating",
    ascending=False
)

,movieId,title,genres,new_user_rating
192,2959,Fight Club (1999),Action|Crime|Drama|Thriller,5.0
166,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,5.0
232,318,"Shawshank Redemption, The (1994)",Crime|Drama,5.0
15,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.5
244,79132,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,4.5
68,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,4.5
16,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.0
20,356,Forrest Gump (1994),Comedy|Drama|Romance|War,3.5
26,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,3.0
987,1721,Titanic (1997),Drama|Romance,2.0


In [16]:
NEW_USER_ID = 999999

In [17]:
def create_new_user_data(
    user_ratings,
    new_user_id=999999
):
    """
    Create rating rows for a new user from
    a dictionary of movieId: rating.
    """

    valid_movie_ids = set(
        movie_catalog["movieId"]
    )

    rows = []


    for movie_id, rating in (
        user_ratings.items()
    ):

        if movie_id not in valid_movie_ids:

            print(
                f"Movie ID {movie_id} "
                f"does not exist and was skipped."
            )

            continue


        if rating < 0.5 or rating > 5:

            print(
                f"Invalid rating for movie "
                f"{movie_id}: {rating}"
            )

            continue


        movie_data = (
            movie_ratings[
                movie_ratings["movieId"]
                == movie_id
            ]
            .iloc[0]
            .copy()
        )


        movie_data["userId"] = (
            new_user_id
        )

        movie_data["rating"] = (
            float(rating)
        )


        rows.append(
            movie_data
        )


    if not rows:

        raise ValueError(
            "No valid movie ratings were supplied."
        )


    return pd.DataFrame(
        rows
    )

In [18]:
new_user_rows = (
    create_new_user_data(
        NEW_USER_RATINGS,
        NEW_USER_ID
    )
)

new_user_rows[
    [
        "userId",
        "movieId",
        "title",
        "rating"
    ]
]

,userId,movieId,title,rating
166,999999,2571,"Matrix, The (1999)",5.0
192,999999,2959,Fight Club (1999),5.0
246,999999,79132,Inception (2010),4.5
15,999999,260,Star Wars: Episode IV - A New Hope (1977),4.5
68,999999,1196,Star Wars: Episode V - The Empire Strikes Back...,4.5
16,999999,296,Pulp Fiction (1994),4.0
232,999999,318,"Shawshank Redemption, The (1994)",5.0
20,999999,356,Forrest Gump (1994),3.5
26,999999,480,Jurassic Park (1993),3.0
1313,999999,1721,Titanic (1997),2.0


In [19]:
demo_ratings = pd.concat(
    [
        movie_ratings,
        new_user_rows
    ],
    ignore_index=True
)

print(
    "Original users:",
    movie_ratings["userId"].nunique()
)

print(
    "Users including new user:",
    demo_ratings["userId"].nunique()
)

Original users: 610
Users including new user: 611


In [20]:
def build_user_genre_preferences(
    ratings
):
    """
    Calculate relative genre preferences
    for each user.
    """

    genre_data = (
        ratings[
            [
                "userId",
                "genres",
                "rating"
            ]
        ]
        .copy()
    )

    genre_data["genre"] = (
        genre_data["genres"]
        .str.split("|")
    )

    genre_data = (
        genre_data
        .explode("genre")
    )


    user_average = (
        ratings
        .groupby("userId")["rating"]
        .mean()
    )


    genre_average = (
        genre_data
        .groupby(
            [
                "userId",
                "genre"
            ]
        )["rating"]
        .mean()
        .unstack()
    )


    preferences = (
        genre_average
        .sub(
            user_average,
            axis=0
        )
        .fillna(0)
    )


    if "(no genres listed)" in (
        preferences.columns
    ):

        preferences = (
            preferences.drop(
                columns="(no genres listed)"
            )
        )


    return preferences

In [21]:
demo_genre_preferences = (
    build_user_genre_preferences(
        demo_ratings
    )
)

In [22]:
existing_preferences = (
    demo_genre_preferences
    .drop(
        index=NEW_USER_ID
    )
)

cluster_profile_data = (
    existing_preferences
    .join(
        user_clusters
        .set_index("userId")
    )
)

cluster_centers = (
    cluster_profile_data
    .groupby("cluster")
    .mean()
)

cluster_centers

,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
cluster,,,,,,,,,,,,,,,,,,,
0,-0.294615,-0.302691,0.007987,-0.118962,0.000487,0.124463,-0.059267,0.180291,-0.250252,0.174620,0.025471,-0.309605,0.104718,0.169047,0.083797,-0.357188,-0.075362,0.230149,-0.022114
1,0.080712,0.090283,0.070783,-0.062720,-0.161606,0.120581,-0.026811,0.030117,-0.054776,-0.038774,-0.027991,0.204244,-0.008880,0.134477,-0.160589,0.122334,0.089655,0.091030,0.003216
2,-0.137340,-0.004482,0.227061,0.008881,-0.078409,0.148508,0.344429,0.172658,0.069804,0.323393,-0.516328,0.322989,0.190612,0.099461,0.101855,-0.227550,-0.064344,0.386132,0.054648
3,-0.088762,-0.234541,-1.057736,-1.098962,-0.294881,0.284867,0.049640,0.165811,-0.515831,0.085993,0.046307,0.002350,-0.897430,0.296880,-0.176026,-0.069192,0.192993,0.113847,0.026787
4,-0.207944,0.292858,0.347612,0.293516,0.159834,-0.392061,-0.014000,-0.110074,0.375500,-0.247982,-0.418799,0.233349,0.141099,-0.496843,0.122643,-0.243823,-0.536153,0.010649,-0.009533


In [23]:
def assign_new_user_cluster(
    user_id,
    genre_preferences,
    cluster_centers
):
    """
    Assign a new user to the closest
    existing cluster using Euclidean distance.
    """

    user_profile = (
        genre_preferences
        .loc[user_id]
    )


    distances = {}


    for cluster_id in (
        cluster_centers.index
    ):

        cluster_profile = (
            cluster_centers
            .loc[cluster_id]
        )


        distance = np.linalg.norm(
            user_profile.values
            -
            cluster_profile.values
        )


        distances[
            cluster_id
        ] = distance


    closest_cluster = min(
        distances,
        key=distances.get
    )


    return (
        closest_cluster,
        distances
    )

In [24]:
new_user_cluster, cluster_distances = (
    assign_new_user_cluster(
        NEW_USER_ID,
        demo_genre_preferences,
        cluster_centers
    )
)

print(
    f"New user assigned to cluster: "
    f"{new_user_cluster}"
)

print(
    "Distances:"
)

for cluster_id, distance in (
    cluster_distances.items()
):

    print(
        f"Cluster {cluster_id}: "
        f"{distance:.3f}"
    )

New user assigned to cluster: 1
Distances:
Cluster 0: 2.142
Cluster 1: 1.530
Cluster 2: 2.126
Cluster 3: 2.419
Cluster 4: 2.500


In [25]:
demo_user_clusters = pd.concat(
    [
        user_clusters,

        pd.DataFrame(
            {
                "userId": [
                    NEW_USER_ID
                ],

                "cluster": [
                    new_user_cluster
                ]
            }
        )
    ],
    ignore_index=True
)

In [26]:
new_content_engine = (
    ContentBasedRecommender(
        demo_ratings,
        movie_catalog
    )
)


new_collaborative_engine = (
    CollaborativeRecommender(
        demo_ratings,
        movie_catalog
    )
)


new_cluster_engine = (
    ClusterRecommender(
        demo_ratings,
        movie_catalog,
        demo_user_clusters
    )
)


new_explainer = (
    RecommendationExplainer(
        demo_ratings
    )
)

In [27]:
base_genre_preferences = (
    build_user_genre_preferences(
        movie_ratings
    )
)

base_cluster_profiles = (
    base_genre_preferences
    .join(
        user_clusters
        .set_index("userId")
    )
)

base_cluster_centers = (
    base_cluster_profiles
    .groupby("cluster")
    .mean()
)

base_cluster_centers

,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
cluster,,,,,,,,,,,,,,,,,,,
0,-0.294615,-0.302691,0.007987,-0.118962,0.000487,0.124463,-0.059267,0.180291,-0.250252,0.174620,0.025471,-0.309605,0.104718,0.169047,0.083797,-0.357188,-0.075362,0.230149,-0.022114
1,0.080712,0.090283,0.070783,-0.062720,-0.161606,0.120581,-0.026811,0.030117,-0.054776,-0.038774,-0.027991,0.204244,-0.008880,0.134477,-0.160589,0.122334,0.089655,0.091030,0.003216
2,-0.137340,-0.004482,0.227061,0.008881,-0.078409,0.148508,0.344429,0.172658,0.069804,0.323393,-0.516328,0.322989,0.190612,0.099461,0.101855,-0.227550,-0.064344,0.386132,0.054648
3,-0.088762,-0.234541,-1.057736,-1.098962,-0.294881,0.284867,0.049640,0.165811,-0.515831,0.085993,0.046307,0.002350,-0.897430,0.296880,-0.176026,-0.069192,0.192993,0.113847,0.026787
4,-0.207944,0.292858,0.347612,0.293516,0.159834,-0.392061,-0.014000,-0.110074,0.375500,-0.247982,-0.418799,0.233349,0.141099,-0.496843,0.122643,-0.243823,-0.536153,0.010649,-0.009533


In [28]:
def recommend_for_new_user(
    user_ratings,
    n=5,
    new_user_id=999999,
    candidate_pool=100
):
    """
    Generate hybrid recommendations for a new user
    based on manually supplied movie ratings.
    """

    # --------------------------------------
    # 1. Require enough ratings
    # --------------------------------------

    if len(user_ratings) < 5:

        raise ValueError(
            "Please rate at least 5 movies "
            "before generating recommendations."
        )


    # --------------------------------------
    # 2. Create temporary new-user data
    # --------------------------------------

    new_user_rows = (
        create_new_user_data(
            user_ratings,
            new_user_id
        )
    )


    demo_ratings = pd.concat(
        [
            movie_ratings,
            new_user_rows
        ],
        ignore_index=True
    )


    # --------------------------------------
    # 3. Build genre profile
    # --------------------------------------

    demo_preferences = (
        build_user_genre_preferences(
            demo_ratings
        )
    )


    # Make sure feature columns match
    # the existing cluster profiles
    new_user_profile = (
        demo_preferences
        .loc[[new_user_id]]
        .reindex(
            columns=base_cluster_centers.columns,
            fill_value=0
        )
    )


    # --------------------------------------
    # 4. Find nearest cluster
    # --------------------------------------

    distances = {}

    for cluster_id in (
        base_cluster_centers.index
    ):

        cluster_profile = (
            base_cluster_centers
            .loc[cluster_id]
        )

        distance = np.linalg.norm(
            new_user_profile
            .iloc[0]
            .values
            -
            cluster_profile.values
        )

        distances[
            cluster_id
        ] = distance


    new_user_cluster = min(
        distances,
        key=distances.get
    )


    # --------------------------------------
    # 5. Add new user to cluster table
    # --------------------------------------

    demo_user_clusters = pd.concat(
        [
            user_clusters,

            pd.DataFrame(
                {
                    "userId": [
                        new_user_id
                    ],

                    "cluster": [
                        new_user_cluster
                    ]
                }
            )
        ],
        ignore_index=True
    )


    # --------------------------------------
    # 6. Create temporary engines
    # --------------------------------------

    content = ContentBasedRecommender(
        demo_ratings,
        movie_catalog
    )

    collaborative = (
        CollaborativeRecommender(
            demo_ratings,
            movie_catalog
        )
    )

    cluster = ClusterRecommender(
        demo_ratings,
        movie_catalog,
        demo_user_clusters
    )

    new_user_explainer = (
        RecommendationExplainer(
            demo_ratings
        )
    )


    # --------------------------------------
    # 7. Generate candidate recommendations
    # --------------------------------------

    content_results = (
        content.recommend(
            user_id=new_user_id,
            n=candidate_pool,
            min_ratings=20
        )
    )


    # Cold-start users have fewer shared movies,
    # so Pearson requirements are relaxed.
    collaborative_results = (
        collaborative.recommend(
            user_id=new_user_id,
            n=candidate_pool,
            min_common=3,
            n_similar_users=30,
            min_similarity=0.10,
            min_neighbor_ratings=2
        )
    )


    cluster_results = (
        cluster.recommend(
            user_id=new_user_id,
            n=candidate_pool,
            min_cluster_ratings=10
        )
    )


    # --------------------------------------
    # 8. Normalize model scores
    # --------------------------------------

    def prepare_scores(
        dataframe,
        score_column,
        normalized_column
    ):

        if (
            dataframe is None
            or dataframe.empty
            or score_column
            not in dataframe.columns
        ):

            return pd.DataFrame(
                columns=[
                    "movieId",
                    score_column,
                    normalized_column
                ]
            )


        result = (
            dataframe[
                [
                    "movieId",
                    score_column
                ]
            ]
            .copy()
        )


        result[
            normalized_column
        ] = (
            result[
                score_column
            ]
            .rank(
                pct=True
            )
        )


        return result


    content_scores = prepare_scores(
        content_results,
        "content_score",
        "content_normalized"
    )

    collaborative_scores = prepare_scores(
        collaborative_results,
        "collaborative_score",
        "collaborative_normalized"
    )

    cluster_scores = prepare_scores(
        cluster_results,
        "cluster_score",
        "cluster_normalized"
    )


    # --------------------------------------
    # 9. Merge the three systems
    # --------------------------------------

    hybrid = (
        content_scores
        .merge(
            collaborative_scores,
            on="movieId",
            how="outer"
        )
        .merge(
            cluster_scores,
            on="movieId",
            how="outer"
        )
    )


    # --------------------------------------
    # 10. Track model support
    # --------------------------------------

    hybrid[
        "content_recommended"
    ] = (
        hybrid["content_score"]
        .notna()
    )

    hybrid[
        "collaborative_recommended"
    ] = (
        hybrid["collaborative_score"]
        .notna()
    )

    hybrid[
        "cluster_recommended"
    ] = (
        hybrid["cluster_score"]
        .notna()
    )


    hybrid[
        "models_recommending"
    ] = (
        hybrid[
            [
                "content_recommended",
                "collaborative_recommended",
                "cluster_recommended"
            ]
        ]
        .sum(axis=1)
    )


    # --------------------------------------
    # 11. Missing normalized scores = 0
    # --------------------------------------

    normalized_columns = [
        "content_normalized",
        "collaborative_normalized",
        "cluster_normalized"
    ]

    hybrid[
        normalized_columns
    ] = (
        hybrid[
            normalized_columns
        ]
        .fillna(0)
    )


    # --------------------------------------
    # 12. Hybrid score
    # --------------------------------------

    hybrid[
        "hybrid_score"
    ] = (
        hybrid[
            "content_normalized"
        ] * 0.40

        +

        hybrid[
            "collaborative_normalized"
        ] * 0.40

        +

        hybrid[
            "cluster_normalized"
        ] * 0.20
    )


    # --------------------------------------
    # 13. Add movie metadata
    # --------------------------------------

    hybrid = (
        hybrid
        .merge(
            movie_catalog,
            on="movieId",
            how="left"
        )
    )


    hybrid = (
        hybrid
        .sort_values(
            [
                "hybrid_score",
                "models_recommending",
                "average_rating"
            ],
            ascending=[
                False,
                False,
                False
            ]
        )
        .head(n)
        .copy()
    )


    # --------------------------------------
    # 14. Generate explanations
    # --------------------------------------

    hybrid[
        "explanation"
    ] = (
        hybrid
        .apply(
            lambda row:
            new_user_explainer
            .generate_explanation(
                user_id=new_user_id,
                recommendation=row
            ),
            axis=1
        )
    )


    # --------------------------------------
    # 15. User genre preferences
    # --------------------------------------

    genre_preferences = (
        content
        .get_genre_preferences(
            new_user_id
        )
        .sort_values(
            [
                "average_rating",
                "movies_rated"
            ],
            ascending=False
        )
    )


    return {
        "recommendations": hybrid,
        "genre_preferences": genre_preferences,
        "cluster": new_user_cluster,
        "cluster_distances": distances
    }

In [29]:
demo_user_widget = (
    widgets.BoundedIntText(
        value=1,
        min=int(
            movie_ratings[
                "userId"
            ].min()
        ),
        max=int(
            movie_ratings[
                "userId"
            ].max()
        ),
        description="User ID:"
    )
)


demo_count_widget = (
    widgets.IntSlider(
        value=5,
        min=3,
        max=10,
        step=1,
        description="Movies:"
    )
)


demo_button = (
    widgets.Button(
        description="Recommend Movies",
        button_style="primary",
        icon="film"
    )
)


demo_output = (
    widgets.Output()
)


def run_demo_mode(button):

    with demo_output:

        clear_output(
            wait=True
        )

        recommend_movies_for_user(
            user_id=demo_user_widget.value,
            n=demo_count_widget.value
        )


demo_button.on_click(
    run_demo_mode
)


demo_mode = widgets.VBox(
    [
        widgets.HTML(
            """
            <h3>Demo Mode</h3>
            <p>
            Select an existing MovieLens user and
            see recommendations based on their
            historical ratings.
            </p>
            """
        ),

        demo_user_widget,
        demo_count_widget,
        demo_button,
        demo_output
    ]
)

In [30]:
try_user_ratings = {}

In [ ]:
movie_search_widget = (
    widgets.Text(
        placeholder="Search for a movie...",
        description="Movie:"
    )
)


search_button = (
    widgets.Button(
        description="Search",
        icon="search"
    )
)


search_results_widget = (
    widgets.Dropdown(
        options=[],
        description="Results:",
        layout=widgets.Layout(
            width="90%"
        )
    )
)


rating_widget = (
    widgets.FloatSlider(
        value=4.0,
        min=0.5,
        max=5.0,
        step=0.5,
        description="Rating:"
    )
)


add_rating_button = (
    widgets.Button(
        description="Add Rating",
        button_style="success",
        icon="plus"
    )
)


rated_movies_widget = (
    widgets.Dropdown(
        options=[],
        description="Rated:",
        layout=widgets.Layout(
            width="90%"
        )
    )
)


remove_rating_button = (
    widgets.Button(
        description="Remove",
        button_style="warning",
        icon="trash"
    )
)


clear_ratings_button = (
    widgets.Button(
        description="Clear All",
        button_style="danger"
    )
)


new_user_count_widget = (
    widgets.IntSlider(
        value=5,
        min=3,
        max=10,
        step=1,
        description="Movies:"
    )
)


generate_button = (
    widgets.Button(
        description="Generate Recommendations",
        button_style="primary",
        icon="film",
        disabled=True
    )
)


rating_status = (
    widgets.HTML(
        value=(
            "<b>0 movies rated.</b> "
            "Rate at least 5 movies."
        )
    )
)


rating_list_output = (
    widgets.Output()
)


new_user_output = (
    widgets.Output()
)

In [32]:
def run_movie_search(button):

    search_text = (
        movie_search_widget
        .value
        .strip()
    )


    if not search_text:

        search_results_widget.options = []

        return


    matches = search_movies(
        search_text,
        n=15
    )


    options = []


    for _, movie in (
        matches.iterrows()
    ):

        label = (
            f"{movie['title']} "
            f"— {movie['genres']}"
        )

        options.append(
            (
                label,
                int(movie["movieId"])
            )
        )


    search_results_widget.options = (
        options
    )


search_button.on_click(
    run_movie_search
)

In [33]:
def search_on_enter(widget):

    run_movie_search(
        None
    )


movie_search_widget.on_submit(
    search_on_enter
)

C:\Users\elemk\AppData\Local\Temp\ipykernel_9920\3596117747.py:8: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  movie_search_widget.on_submit(


In [34]:
def refresh_new_user_ratings():

    # ----------------------------------
    # Status
    # ----------------------------------

    count = len(
        try_user_ratings
    )


    if count < 5:

        rating_status.value = (
            f"<b>{count} movies rated.</b> "
            f"Rate at least {5 - count} more."
        )

        generate_button.disabled = True

    else:

        rating_status.value = (
            f"<b>{count} movies rated.</b> "
            "Ready for recommendations!"
        )

        generate_button.disabled = False


    # ----------------------------------
    # Rated movie dropdown
    # ----------------------------------

    rated_options = []


    for movie_id, rating in (
        try_user_ratings.items()
    ):

        movie = (
            movie_catalog[
                movie_catalog["movieId"]
                == movie_id
            ]
            .iloc[0]
        )


        rated_options.append(
            (
                f"{movie['title']} "
                f"— {rating:.1f}/5",
                movie_id
            )
        )


    rated_movies_widget.options = (
        rated_options
    )


    # ----------------------------------
    # Table display
    # ----------------------------------

    with rating_list_output:

        clear_output(
            wait=True
        )


        if not try_user_ratings:

            print(
                "No movies rated yet."
            )

            return


        selected = (
            movie_catalog[
                movie_catalog["movieId"]
                .isin(
                    try_user_ratings.keys()
                )
            ][
                [
                    "movieId",
                    "title",
                    "genres"
                ]
            ]
            .copy()
        )


        selected[
            "Your Rating"
        ] = (
            selected["movieId"]
            .map(
                try_user_ratings
            )
        )


        display(
            selected[
                [
                    "title",
                    "genres",
                    "Your Rating"
                ]
            ]
            .sort_values(
                "Your Rating",
                ascending=False
            )
        )

In [35]:
def add_movie_rating(button):

    movie_id = (
        search_results_widget.value
    )


    if movie_id is None:

        return


    try_user_ratings[
        int(movie_id)
    ] = float(
        rating_widget.value
    )


    refresh_new_user_ratings()


add_rating_button.on_click(
    add_movie_rating
)

In [36]:
def remove_movie_rating(button):

    movie_id = (
        rated_movies_widget.value
    )


    if movie_id is None:

        return


    try_user_ratings.pop(
        int(movie_id),
        None
    )


    refresh_new_user_ratings()


remove_rating_button.on_click(
    remove_movie_rating
)

In [37]:
def clear_all_ratings(button):

    try_user_ratings.clear()

    refresh_new_user_ratings()


    with new_user_output:

        clear_output()


clear_ratings_button.on_click(
    clear_all_ratings
)

In [38]:
def run_new_user_recommendation(
    button
):

    with new_user_output:

        clear_output(
            wait=True
        )


        try:

            result = (
                recommend_for_new_user(
                    user_ratings=
                        try_user_ratings,

                    n=
                        new_user_count_widget
                        .value
                )
            )


        except Exception as error:

            print(
                f"Unable to generate "
                f"recommendations: {error}"
            )

            return


        recommendations = (
            result[
                "recommendations"
            ]
        )

        genre_preferences = (
            result[
                "genre_preferences"
            ]
        )

        cluster = (
            result[
                "cluster"
            ]
        )


        print(
            "=" * 75
        )

        print(
            "YOUR PERSONALIZED "
            "MOVIE RECOMMENDATIONS"
        )

        print(
            "=" * 75
        )


        print(
            f"\nMovies rated: "
            f"{len(try_user_ratings)}"
        )

        print(
            f"Preference cluster: "
            f"{cluster}"
        )


        print(
            "\nTOP INFERRED GENRES"
        )

        print(
            "-" * 75
        )


        for genre, row in (
            genre_preferences
            .head(5)
            .iterrows()
        ):

            print(
                f"{genre:<15} "
                f"{row['average_rating']:.2f}/5 "
                f"({int(row['movies_rated'])} movies)"
            )


        print(
            "\nTOP RECOMMENDATIONS"
        )

        print(
            "=" * 75
        )


        for position, (_, movie) in (
            enumerate(
                recommendations
                .iterrows(),
                start=1
            )
        ):

            print(
                f"\n{position}. "
                f"{movie['title']}"
            )

            print(
                f"   Genres: "
                f"{movie['genres']}"
            )

            print(
                f"   Director: "
                f"{movie['director']}"
            )

            print(
                f"   Hybrid score: "
                f"{movie['hybrid_score']:.3f}"
            )

            print(
                f"   Models supporting: "
                f"{int(movie['models_recommending'])}/3"
            )

            print(
                f"   Why: "
                f"{movie['explanation']}"
            )


        print(
            "\n" + "=" * 75
        )


generate_button.on_click(
    run_new_user_recommendation
)

In [39]:
try_yourself_mode = (
    widgets.VBox(
        [
            widgets.HTML(
                """
                <h3>Try It Yourself</h3>

                <p>
                Search for movies you have seen,
                give each one a rating from
                <b>0.5 to 5</b>, and the system
                will build a personalized profile
                for you.
                </p>

                <p>
                Rate at least <b>5 movies</b>.
                More ratings usually produce
                better recommendations.
                </p>
                """
            ),

            widgets.HBox(
                [
                    movie_search_widget,
                    search_button
                ]
            ),

            search_results_widget,

            widgets.HBox(
                [
                    rating_widget,
                    add_rating_button
                ]
            ),

            rating_status,

            rated_movies_widget,

            widgets.HBox(
                [
                    remove_rating_button,
                    clear_ratings_button
                ]
            ),

            rating_list_output,

            widgets.HTML(
                "<hr>"
            ),

            new_user_count_widget,

            generate_button,

            new_user_output
        ]
    )
)

In [40]:
recommendation_app = (
    widgets.Tab(
        children=[
            demo_mode,
            try_yourself_mode
        ]
    )
)


recommendation_app.set_title(
    0,
    "Demo Mode"
)

recommendation_app.set_title(
    1,
    "Try It Yourself"
)


display(
    widgets.HTML(
        """
        <h1>🎬 AI-Powered Movie Recommendation System</h1>

        <p>
        Explore recommendations for an existing
        MovieLens user or rate movies yourself
        to receive personalized suggestions.
        </p>
        """
    )
)


display(
    recommendation_app
)

HTML(value='\n        <h1>🎬 AI-Powered Movie Recommendation System</h1>\n\n        <p>\n        Explore recomm…